# 08 — Serie temporali III: Box-Jenkins, VAR, GARCH (1970–1986)
Kernel **JupyterNet** (C#) su runtime **.NET 11**. Prerequisito: `dotnet build src/MathemaSharp.slnx -c Release`. Da terminale: `scripts/run-notebooks.ps1` (imposta il roll-forward al runtime 11); in VS Code serve `DOTNET_ROLL_FORWARD=LatestMajor` e, finché .NET 11 è RC, `DOTNET_ROLL_FORWARD_TO_PRERELEASE=1`.

**Box-Jenkins (1970)**: identificazione sistematica, stima e controllo dei residui. **VAR (Sims, 1980)**: più serie, ciascuna spiegata dai propri ritardi e da quelli delle altre. **ARCH/GARCH (Engle 1982, Bollerslev 1986)**: si modella la *varianza* condizionale, non la media.

In [ ]:
#load "../shared/MathemaSharp.csx"
using System.Numerics;
using MathemaSharp.Core;
using MathemaSharp.TimeSeries;
// ARIMA(1,1,1): le differenze seguono un ARMA(1,1) con φ = 0.5, θ = 0.4; la serie è la loro somma cumulata.
var rng = new Random(1970);
var w = Simulation.Arma<double>(rng, 400, 0.2, new[] { 0.5 }, new[] { 0.4 }, 1);
var y = new double[w.Length + 1];
y[0] = 100;
for (var t = 0; t < w.Length; t++) y[t + 1] = y[t] + w[t];
var dy = SeriesStatistics.Difference(y);
var band = 1.96 / Math.Sqrt(dy.Length);
Chart.Lines("Serie integrata: non stazionaria in media", new Line("y", y));
Chart.Stems("ACF di y: decadimento lentissimo ⇒ differenziare", SeriesStatistics.Acf(y, 30)[1..], 1.96 / Math.Sqrt(y.Length));
Chart.Stems("ACF di Δy", SeriesStatistics.Acf(dy, 30)[1..], band);
Chart.Stems("PACF di Δy", SeriesStatistics.Pacf(dy, 30), band);

## MA = innovazioni ritardate, non media mobile
Nel modello $w_t = c+\phi w_{t-1}+e_t+\theta e_{t-1}$ il termine MA è lo shock di ieri, non una media mobile dei dati. Lo si vede dalla rappresentazione di Wold: i pesi $\psi_j$ della risposta a un impulso seguono $\psi_0=1$, $\psi_j=\theta_j+\sum_i\phi_i\psi_{j-i}$. Per ARMA(1,1): $\psi_1=\phi+\theta$, poi decadimento geometrico con ragione $\phi$.

In [ ]:
double[] Psi(double[] phi, double[] theta, int n)
{
    var psi = new double[n];
    psi[0] = 1;
    for (var j = 1; j < n; j++)
    {
        psi[j] = j <= theta.Length ? theta[j - 1] : 0;
        for (var i = 1; i <= phi.Length && i <= j; i++) psi[j] += phi[i - 1] * psi[j - i];
    }
    return psi;
}
Chart.Stems("Pesi ψ_j di ARMA(1,1), φ=0.5, θ=0.4 (lag 0…15)", Psi(new[] { 0.5 }, new[] { 0.4 }, 16), firstLag: 0);

## Stima e scelta dell'ordine
`FitHannanRissanen` è lineare e senza ottimizzazione:
1. stima un AR lungo con OLS per ricostruire le innovazioni $\hat e_t$;
2. stima con OLS $w_t$ sui $w$ ritardati e sugli $\hat e$ ritardati.

Le regressioni usano la **QR di Householder**: le equazioni normali $X^\top X$ eleverebbero al quadrato il numero di condizionamento. Per confrontare gli ordini si usano un AIC approssimato $n\ln\hat\sigma^2+2k$ e il test di Ljung-Box sui residui.

In [ ]:
var candidates = new[] { (1, 1, 0), (0, 1, 1), (1, 1, 1), (2, 1, 0), (2, 1, 2) };
Table.Show(new[] { "ARIMA", "c", "φ", "θ", "σ²", "AIC≈", "Ljung-Box p (20)" }, candidates.Select(o =>
{
    var (p, d, q) = o;
    var m = ArimaModel.FitHannanRissanen(y, p, d, q);
    var res = m.Residuals(y)[Math.Max(p, q)..];
    var aic = res.Length * Math.Log(m.InnovationVariance) + 2 * (p + q + 1);
    return new object[] { $"({p},{d},{q})", m.Constant, string.Join(" ", m.Phi.Select(v => v.ToString("F3"))), string.Join(" ", m.Theta.Select(v => v.ToString("F3"))), m.InnovationVariance, aic, SeriesStatistics.LjungBox(res, 20, p + q).PValue };
}));

Lettura della tabella: con $n=400$ Hannan-Rissanen è consistente ma rumoroso sui singoli parametri, mentre $\psi_1=\phi+	heta$ (il primo peso della risposta all'impulso) è determinato molto meglio. Modelli più ricchi abbassano $\hat\sigma^2$ di poco: l'AIC penalizza i parametri, Ljung-Box controlla che non resti struttura. La massima verosimiglianza esatta (non implementata qui) partirebbe proprio da queste stime.

In [ ]:
var best = ArimaModel.FitHannanRissanen(y[..380], 1, 1, 1);
var fc = best.Forecast(y[..380], 21);
Chart.Lines("ARIMA(1,1,1): previsione a 21 passi (asse x: t − 340)", new Line("osservato", y[340..]), new Line("previsione", fc, Start: 40, Dashed: true));
Console.WriteLine($"RMSE fuori campione: ARIMA {Accuracy.Rmse(y[380..], fc):F3}; naive {Accuracy.Rmse(y[380..], ReferenceForecasts.Naive(y[..380], 21)):F3}; drift {Accuracy.Rmse(y[380..], ReferenceForecasts.Drift(y[..380], 21)):F3}");

## VAR: più serie, ritardi incrociati
$$Y_t = c + A_1 Y_{t-1} + \dots + A_p Y_{t-p} + \varepsilon_t.$$
Con gli stessi regressori in ogni equazione, l'OLS equazione per equazione coincide con il GLS/SUR (Zellner). Un VAR(1) è stazionario se gli autovalori di $A_1$ stanno dentro il cerchio unitario; la media stazionaria è $\mu=(I-A_1)^{-1}c$. La risposta a un impulso $A_1^h e_j$ mostra come uno shock su una serie si propaga all'altra.

In [ ]:
var a = new double[,] { { 0.5, 0.3 }, { -0.2, 0.6 } };
var trueVar = new VarModel<double>(new[] { 1.0, 0.5 }, new[] { a }, new double[2, 2]);
var data = Simulation.Var(new Random(1980), trueVar, 500, 1);
var fit = VarModel.Fit(data, 1);
Table.Show(new[] { "", "A[0,0]", "A[0,1]", "A[1,0]", "A[1,1]", "c0", "c1" }, new[]
{
    new object[] { "vero", 0.5, 0.3, -0.2, 0.6, 1.0, 0.5 },
    new object[] { "OLS", fit.Coefficients[0][0, 0], fit.Coefficients[0][0, 1], fit.Coefficients[0][1, 0], fit.Coefficients[0][1, 1], fit.Intercept[0], fit.Intercept[1] },
});
Console.WriteLine($"Stabile: {fit.IsStableOrder1Bivariate()}  (autovalori veri 0.55 ± 0.24i, modulo 0.60)");
var x = Enumerable.Range(0, 120).Select(t => data[t, 0]).ToArray();
var z = Enumerable.Range(0, 120).Select(t => data[t, 1]).ToArray();
Chart.Lines("Due serie di un VAR(1) (primi 120 passi)", new Line("x", x), new Line("y", z));
// Risposta a un impulso unitario su x al tempo 0.
var irf = fit.Forecast(new double[,] { { 1 - 0, 0 } }, 15);
var mu = fit.Forecast(new double[,] { { 0, 0 } }, 15);
Chart.Lines("Risposta a uno shock unitario su x", new Line("x", Enumerable.Range(0, 15).Select(h => irf[h, 0] - mu[h, 0]).ToArray()), new Line("y", Enumerable.Range(0, 15).Select(h => irf[h, 1] - mu[h, 1]).ToArray()));

## ARCH/GARCH: volatilità, non media
$$\varepsilon_t=\sigma_t z_t,\qquad \sigma_t^2=\omega+\alpha\varepsilon_{t-1}^2+\beta\sigma_{t-1}^2 .$$
I rendimenti sono quasi incorrelati, ma i loro **quadrati** no: grandi shock seguono grandi shock (volatility clustering). La previsione della varianza decade geometricamente, con ragione $\alpha+\beta$, verso $\bar\sigma^2=\omega/(1-\alpha-\beta)$.

La stima usa il **variance targeting**: si fissa $\omega=s^2(1-\alpha-\beta)$, così la massima verosimiglianza si riduce a due parametri, cercati su una griglia raffinata.

In [ ]:
var garchTrue = new Garch11<double>(0.05, 0.1, 0.85);
var eps = Simulation.Garch(new Random(1986), garchTrue, 3000);
var g = Garch11.FitVarianceTargeting(eps);
var sigma = g.ConditionalVariances(eps).Select(Math.Sqrt).ToArray();
Chart.Panels("Rendimenti simulati e volatilità condizionale stimata", new Line("ε_t", eps[..600]), new Line("σ_t stimata", sigma[..600]));
var b = 1.96 / Math.Sqrt(eps.Length);
Chart.Stems("ACF di ε: quasi nulla", SeriesStatistics.Acf(eps, 20)[1..], b);
Chart.Stems("ACF di ε²: volatility clustering", SeriesStatistics.Acf(eps.Select(e => e * e).ToArray(), 20)[1..], b);
Table.Show(new[] { "", "ω", "α", "β", "α+β", "σ̄²", "log-lik" }, new[]
{
    new object[] { "vero", garchTrue.Omega, garchTrue.Alpha, garchTrue.Beta, garchTrue.Persistence, garchTrue.UnconditionalVariance, garchTrue.LogLikelihood(eps) },
    new object[] { "stimato", g.Omega, g.Alpha, g.Beta, g.Persistence, g.UnconditionalVariance, g.LogLikelihood(eps) },
});
Chart.Lines("Previsione di σ² verso la varianza incondizionata", new Line("σ²_{T+h}", g.ForecastVariance(eps, 100)), new Line("σ̄²", Enumerable.Repeat(g.UnconditionalVariance, 100).ToArray(), Dashed: true));